# Lexicon Exploration — Phase 2 Datasets

Exploring the downloaded idiom and phrasal verb lexicons:

| File | Source | Description |
|------|--------|-------------|
| `phrasal_verbs_semigradsky.json` | Semigradsky/phrasal-verbs | 124 phrasal verbs with separability markers |
| `phrasal_verbs_wecan.json` | WithEnglishWeCan | 3,350 phrasal verbs with conjugations |
| `idioms_mcgrawhill.json` | zaghloul404/englishidioms | 21,866 idiom entries from McGraw-Hill |
| `magpie_idioms.jsonl` | MAGPIE Corpus (CC-BY-4.0) | 56,622 instances of 1,756 idioms in context |

In [7]:
import json
from collections import Counter
from pathlib import Path

LEXICON_DIR = Path('../data/lexicons')

def load_json(name):
    with open(LEXICON_DIR / name) as f:
        return json.load(f)

def load_jsonl(name):
    with open(LEXICON_DIR / name) as f:
        return [json.loads(line) for line in f]

---
## 1. Semigradsky Phrasal Verbs (124 entries)

A curated list of common phrasal verbs. Uses `*` and `+` markers to indicate object placement and separability.

In [8]:
semi = load_json('phrasal_verbs_semigradsky.json')
print(f"Total entries: {len(semi)}")
print(f"Fields: {list(semi[0].keys())}")
print()

# Show first 5 entries
for e in semi[:5]:
    print(f"  {e['verb']:<25} {e['definition'][:60]}")
    print(f"  {'':25} ex: {e['examples'][0][:70]}")
    print()

Total entries: 124
Fields: ['verb', 'definition', 'examples']

  blow * up +               make explode;destroy using explosives
                            ex: The terrorists blew the bridge up.

  blow up                   suddenly become very angry
                            ex: When Joan heard the news, she blew up and rushed out of the room.

  break * down +            analyze in detail
                            ex: We need to break this problem down in order to solve.

  break down                stop working properly
                            ex: The truck broke down in the desert.

  break down                become mentally ill
                            ex: She broke down after her husband died.



In [9]:
# Analyze separability markers
# * = object can go between verb and particle
# + = requires an object
separable = [e for e in semi if '*' in e['verb']]
requires_obj = [e for e in semi if '+' in e['verb']]
intransitive = [e for e in semi if '*' not in e['verb'] and '+' not in e['verb']]

print(f"Separable (has *):       {len(separable)}")
print(f"Requires object (has +): {len(requires_obj)}")
print(f"Intransitive (no marks): {len(intransitive)}")
print()

# Extract base verb + particle
def parse_semi_verb(v):
    parts = v.replace('*', '').replace('+', '').strip().split()
    return parts[0] if parts else '', ' '.join(parts[1:]) if len(parts) > 1 else ''

verbs = Counter(parse_semi_verb(e['verb'])[0] for e in semi)
print("Most common base verbs:")
for verb, count in verbs.most_common(10):
    print(f"  {verb:<12} {count}")

Separable (has *):       42
Requires object (has +): 57
Intransitive (no marks): 58

Most common base verbs:
  go           18
  take         15
  come         14
  get          9
  bring        7
  look         6
  put          6
  turn         6
  call         5
  make         5


In [10]:
# Show some intransitive vs separable examples
print("Intransitive examples:")
for e in intransitive[:5]:
    print(f"  {e['verb']:<20} - {e['definition'][:50]}")

print("\nSeparable examples:")
for e in separable[:5]:
    print(f"  {e['verb']:<20} - {e['definition'][:50]}")

Intransitive examples:
  blow up              - suddenly become very angry
  break down           - stop working properly
  break down           - become mentally ill
  bring back           - return
  bring back           - return to consciousness

Separable examples:
  blow * up +          - make explode;destroy using explosives
  break * down +       - analyze in detail
  bring * up +         - mention
  bring * up +         - raise (a child)
  call * off +         - cancel something


---
## 2. WithEnglishWeCan Phrasal Verbs (3,350 entries)

A large generated dataset with conjugated forms, definitions, examples, and synonyms.

In [11]:
wecan = load_json('phrasal_verbs_wecan.json')
print(f"Total phrasal verbs: {len(wecan)}")

first_key = list(wecan.keys())[0]
print(f"Fields per entry: {list(wecan[first_key].keys())}")
print()

# Show a few entries
for key in list(wecan.keys())[:5]:
    entry = wecan[key]
    print(f"  {key}")
    print(f"    Derivatives: {entry.get('derivatives', [])}")
    print(f"    Definition:  {entry['descriptions'][0][:70] if entry.get('descriptions') else '(none)'}")
    print(f"    Synonyms:    {entry.get('synonyms', [])[:5]}")
    print()

Total phrasal verbs: 3350
Fields per entry: ['derivatives', 'descriptions', 'examples', 'frequency', 'synonyms', 'translations']

  abide by
    Derivatives: ['abides by', 'abiding by', 'abided by']
    Definition:  to follow a rule, decision, or instruction
    Synonyms:    ['agree to', 'carry out', 'follow', 'obey']

  accord with
    Derivatives: ['accords with', 'according with', 'accorded with']
    Definition:  to agree with or be the same as something else
    Synonyms:    []

  account for
    Derivatives: ['accounts for', 'accounting for', 'accounted for']
    Definition:  if someone is accounted for, you know where they are and so do not wor
    Synonyms:    ['destroy', 'kill [informal]', 'put out of action', 'put paid to']

  ache for
    Derivatives: []
    Definition:  to want something or someone very much
    Synonyms:    []

  act as
    Derivatives: ['acts as', 'acting as', 'acted as']
    Definition:  to do the job of a particular kind of person or thing
    Synonyms:

In [12]:
# Extract particles used
particles = Counter()
base_verbs = Counter()
for pv in wecan.keys():
    parts = pv.split()
    if len(parts) >= 2:
        base_verbs[parts[0]] += 1
        particles[' '.join(parts[1:])] += 1

print(f"Unique base verbs: {len(base_verbs)}")
print(f"Unique particles:  {len(particles)}")
print()
print("Top 15 particles:")
for particle, count in particles.most_common(15):
    print(f"  {particle:<15} {count:>4}")

print("\nTop 15 base verbs:")
for verb, count in base_verbs.most_common(15):
    print(f"  {verb:<15} {count:>4}")

Unique base verbs: 1166
Unique particles:  196

Top 15 particles:
  up               464
  out              402
  off              250
  in               193
  on               191
  down             173
  away             112
  around           105
  over             105
  into             105
  back              79
  with              74
  through           73
  for               72
  to                64

Top 15 base verbs:
  get               69
  go                60
  come              54
  put               32
  be                27
  run               27
  take              24
  look              23
  give              22
  hold              20
  bring             19
  stand             19
  fall              18
  pull              18
  hang              17


In [13]:
# How many have derivatives (conjugated forms)?
has_derivatives = sum(1 for e in wecan.values() if e.get('derivatives'))
has_examples = sum(1 for e in wecan.values() if e.get('examples'))
has_synonyms = sum(1 for e in wecan.values() if e.get('synonyms'))

print(f"Have derivatives: {has_derivatives}/{len(wecan)} ({has_derivatives*100/len(wecan):.0f}%)")
print(f"Have examples:   {has_examples}/{len(wecan)} ({has_examples*100/len(wecan):.0f}%)")
print(f"Have synonyms:   {has_synonyms}/{len(wecan)} ({has_synonyms*100/len(wecan):.0f}%)")

# Sample entry with all fields populated
print("\nFull sample entry:")
for key, entry in wecan.items():
    if entry.get('derivatives') and entry.get('synonyms') and entry.get('examples'):
        print(json.dumps({key: entry}, indent=2)[:600])
        break

Have derivatives: 2660/3350 (79%)
Have examples:   2343/3350 (70%)
Have synonyms:   648/3350 (19%)

Full sample entry:
{
  "abide by": {
    "derivatives": [
      "abides by",
      "abiding by",
      "abided by"
    ],
    "descriptions": [
      "to follow a rule, decision, or instruction"
    ],
    "examples": [
      "Ah, surely,' Maram said, `this is the time to abide by the letter of his rule?",
      "They promised to abide by the rules of the contest."
    ],
    "frequency": 0,
    "synonyms": [
      "agree to",
      "carry out",
      "follow",
      "obey"
    ],
    "translations": {
      "2": {
        "\u0431\u043b\u044e\u0441\u0442\u0438": [],
        "\u0432\u044b\u043f\u043e\u043b\u043d\


In [14]:
# Find phrasal verbs for a specific base verb
def find_phrasal_verbs(base):
    return {k: v for k, v in wecan.items() if k.startswith(base + ' ')}

for verb in ['get', 'take', 'put', 'turn']:
    pvs = find_phrasal_verbs(verb)
    print(f"\n'{verb}' phrasal verbs ({len(pvs)}):")
    for pv in sorted(pvs.keys())[:8]:
        desc = pvs[pv]['descriptions'][0][:50] if pvs[pv].get('descriptions') else ''
        print(f"  {pv:<20} {desc}")
    if len(pvs) > 8:
        print(f"  ... and {len(pvs) - 8} more")


'get' phrasal verbs (69):
  get about            same as get around
  get above            behave as if you are better or more important than
  get across           to make people understand something
  get across to        be convincing or make a good impression
  get after            to chase someone or something
  get ahead            to be more successful, or to progress more quickly
  get ahead of         move in front of
  get along            if people get along, they like each other and are 
  ... and 61 more

'take' phrasal verbs (24):
  take aback           surprise or shock
  take after           to look or behave like an older relative
  take against         to begin to dislike someone, often without having 
  take apart           to beat someone very easily in a game or sport
  take aside           to take someone away from someone else they are wi
  take away            to reduce the positive effect or success of someth
  take away from       to reduce the positive effec

In [44]:
# Find phrasal verbs for a specific base verb
def find_phrasal_verbs(base):
    return {k: v for k, v in wecan.items() if k.startswith(base + ' ')}

for verb in ['screw']:
    pvs = find_phrasal_verbs(verb)
    print(f"\n'{verb}' phrasal verbs ({len(pvs)}):")
    for pv in sorted(pvs.keys())[:8]:
        desc = pvs[pv]['descriptions'][0][:50] if pvs[pv].get('descriptions') else ''
        print(f"  {pv:<20} {desc}")
    if len(pvs) > 8:
        print(f"  ... and {len(pvs) - 8} more")


'screw' phrasal verbs (3):
  screw around         to have a lot of sexual relationships, instead of 
  screw over           to deliberately put someone in an unfavorable situ
  screw up             if you screw up your eyes, you close them tightly


---
## 3. McGraw-Hill Idioms (21,866 entries)

Comprehensive dictionary of American idioms and phrasal verbs. Each entry includes the phrase, definition, alternate forms, patterns, and word forms.

In [39]:
mcgraw_raw = load_json('idioms_mcgrawhill.json')
mcgraw = mcgraw_raw['dictionary']
print(f"Total entries: {len(mcgraw)}")
print(f"Fields: {list(mcgraw[0].keys())}")
print()

# Show a few entries
for e in mcgraw[:3]:
    print(f"  [{e['id']}] {e['phrase']}")
    print(f"    Definition: {e['definition'][:120]}...")
    print(f"    Patterns:   {e.get('patterns', [])[:3]}")
    print(f"    Alt forms:  {e.get('alt', [])[:3]}")
    print(f"    Word forms: {e.get('word_forms', [])[:3]}")
    print()

Total entries: 21866
Fields: ['id', 'range', 'phrase', 'phrase_html', 'definition', 'definition_html', 'alt', 'runs', 'patterns', 'word_forms', 'multiple', 'duplicate']

  [0] able to breathe (easily) again
    Definition: 1. Lit. able to breathe clean, fresh air with no restriction or obstruction. _After I got out of the dank basement, I wa...
    Patterns:   ['able to breathe again', 'able to breathe easily again']
    Alt forms:  ['constant', 'o-constant', 'constant']
    Word forms: [[['ability', 'ably', 'abilities', 'able'], ['to'], ['breathe', 'breathes', 'breathers', 'breathings', 'breather', 'breathing', 'breathed']], [['easiness', 'easily', 'easinesses', 'easy']], [['again']]]

  [1] able to breathe (freely) again
    Definition: 1. Lit. able to breathe clean, fresh air with no restriction or obstruction. _After I got out of the dank basement, I wa...
    Patterns:   ['able to breathe again', 'able to breathe freely again']
    Alt forms:  ['constant', 'o-constant', 'constant'

In [16]:
# How many unique phrases (vs duplicates)?
unique_phrases = set(e['phrase'].lower() for e in mcgraw)
print(f"Total entries:    {len(mcgraw)}")
print(f"Unique phrases:   {len(unique_phrases)}")
print(f"Duplicates:       {len(mcgraw) - len(unique_phrases)}")

# Count by word count in phrase
word_counts = Counter(len(e['phrase'].split()) for e in mcgraw)
print("\nPhrase length distribution:")
for wc in sorted(word_counts.keys())[:10]:
    bar = '#' * (word_counts[wc] // 100)
    print(f"  {wc} words: {word_counts[wc]:>5}  {bar}")

Total entries:    21866
Unique phrases:   21253
Duplicates:       613

Phrase length distribution:
  1 words:   150  #
  2 words:  2294  ######################
  3 words:  6024  ############################################################
  4 words:  4750  ###############################################
  5 words:  3967  #######################################
  6 words:  2485  ########################
  7 words:  1191  ###########
  8 words:   556  #####
  9 words:   236  ##
  10 words:   127  #


In [40]:
# Entries with patterns (regex-like variant matching)
has_patterns = [e for e in mcgraw if e.get('patterns')]
has_alt = [e for e in mcgraw if e.get('alt')]
has_word_forms = [e for e in mcgraw if e.get('word_forms')]
has_runs = [e for e in mcgraw if e.get('runs')]

print(f"Have patterns:    {len(has_patterns):>6} ({len(has_patterns)*100/len(mcgraw):.0f}%)")
print(f"Have alt forms:   {len(has_alt):>6} ({len(has_alt)*100/len(mcgraw):.0f}%)")
print(f"Have word_forms:  {len(has_word_forms):>6} ({len(has_word_forms)*100/len(mcgraw):.0f}%)")
print(f"Have runs:        {len(has_runs):>6} ({len(has_runs)*100/len(mcgraw):.0f}%)")

Have patterns:     21866 (100%)
Have alt forms:    21866 (100%)
Have word_forms:   21866 (100%)
Have runs:         21866 (100%)


In [18]:
# Look at pattern structure
print("Sample entries with patterns:")
for e in has_patterns[:5]:
    print(f"\n  Phrase: {e['phrase']}")
    print(f"  Patterns: {e['patterns'][:3]}")

Sample entries with patterns:

  Phrase: able to breathe (easily) again
  Patterns: ['able to breathe again', 'able to breathe easily again']

  Phrase: able to breathe (freely) again
  Patterns: ['able to breathe again', 'able to breathe freely again']

  Phrase: able to breathe (freely) again
  Patterns: ['can breathe again', 'can breathe freely again']

  Phrase: able to do something blindfolded
  Patterns: ['able to blindfolded']

  Phrase: able to do something standing on one’s head
  Patterns: ['able to standing on head']


In [19]:
# Look at word_forms structure
print("Sample entries with word_forms:")
for e in has_word_forms[:5]:
    print(f"\n  Phrase:     {e['phrase']}")
    print(f"  Word forms: {e['word_forms'][:5]}")

Sample entries with word_forms:

  Phrase:     able to breathe (easily) again
  Word forms: [[['ability', 'ably', 'abilities', 'able'], ['to'], ['breathe', 'breathes', 'breathers', 'breathings', 'breather', 'breathing', 'breathed']], [['easiness', 'easily', 'easinesses', 'easy']], [['again']]]

  Phrase:     able to breathe (freely) again
  Word forms: [[['ability', 'ably', 'abilities', 'able'], ['to'], ['breathe', 'breathes', 'breathers', 'breathings', 'breather', 'breathing', 'breathed']], [['frees', 'freeing', 'freeings', 'freed', 'freely', 'free']], [['again']]]

  Phrase:     able to breathe (freely) again
  Word forms: [[['can', 'cannery', 'cans', 'canneries', "can't", "couldn't", 'could'], ['breathe', 'breathes', 'breathers', 'breathings', 'breather', 'breathing', 'breathed']], [['frees', 'freeing', 'freeings', 'freed', 'freely', 'free']], [['again']]]

  Phrase:     able to do something blindfolded
  Word forms: [[['ability', 'ably', 'abilities', 'able'], ['to']], 'NA', [['blin

In [20]:
# Search the idiom dictionary
def search_mcgraw(keyword):
    keyword = keyword.lower()
    return [e for e in mcgraw if keyword in e['phrase'].lower()]

for term in ['kick', 'bucket', 'break', 'heart']:
    results = search_mcgraw(term)
    print(f"\n'{term}' -> {len(results)} entries:")
    for e in results[:5]:
        print(f"  - {e['phrase']}")
    if len(results) > 5:
        print(f"  ... and {len(results) - 5} more")


'kick' -> 50 entries:
  - alive and kicking
  - kick someone or an animal out†
  - for kicks
  - get a kick out of someone or something
  - give someone a kick
  ... and 45 more

'bucket' -> 7 entries:
  - can’t carry a tune in a bucket
  - a drop in the bucket
  - For crying in a bucket!
  - go to hell in a bucket
  - don’t amount to a bucket of spit
  ... and 2 more

'break' -> 111 entries:
  - at the break of dawn
  - break a habit
  - break the habit
  - break one’s habit
  - break a law
  ... and 106 more

'heart' -> 90 entries:
  - *the heart of the matter
  - cry one’s heart out
  - sing one’s heart out
  - play one’s heart out
  - sob one’s heart out
  ... and 85 more


In [43]:
for term in ['screw']:
    results = search_mcgraw(term)
    print(f"\n'{term}' -> {len(results)} entries:")
    for e in results:
    # for e in results[:5]:
        print(f"  - {e['phrase']}")
    # if len(results) > 5:
    #     print(f"  ... and {len(results) - 5} more")


'screw' -> 22 entries:
  - have a screw loose
  - have a loose screw
  - have got a screw loose
  - put the screws on (someone)
  - put the screws on
  - get screwed
  - screw around
  - screw around with someone or something
  - screw off
  - screw someone around
  - screw someone or something up†
  - screw someone out of something
  - screw someone over†
  - screw someone up†
  - screw something down†
  - screw something into something
  - screw something (on)(to something)
  - screw something up†
  - screw up
  - screw up one’s courage
  - screwed, blued, and tattooed
  - screwed up


---
## 4. MAGPIE Idiom Corpus (56,622 instances / 1,756 idioms)

A sense-annotated corpus of potentially idiomatic expressions from the British National Corpus. Each instance has an idiom used in context, labeled as idiomatic (`i`), literal (`l`), figurative (`f`), or other.

In [36]:
magpie = load_jsonl('magpie_idioms.jsonl')
print(f"Total instances: {len(magpie)}")
print(f"Fields: {list(magpie[0].keys())}")
print()

# Label distribution
labels = Counter(e['label'] for e in magpie)
label_names = {'i': 'idiomatic', 'l': 'literal', 'f': 'figurative', '?': 'unknown', 'o': 'other'}
print("Label distribution:")
for label, count in labels.most_common():
    name = label_names.get(label, label)
    bar = '#' * (count // 200)
    print(f"  {label} ({name:<12}): {count:>6}  {bar}")

Total instances: 56622
Fields: ['confidence', 'context', 'document_id', 'genre', 'id', 'idiom', 'judgment_count', 'label', 'label_distribution', 'non_standard_usage_explanations', 'offsets', 'sentence_no', 'split', 'variant_type']

Label distribution:
  i (idiomatic   ):  40011  ########################################################################################################################################################################################################
  l (literal     ):  16168  ################################################################################
  o (other       ):    436  ##
  ? (unknown     ):      7  


In [22]:
# Unique idioms
idiom_counts = Counter(e['idiom'] for e in magpie)
print(f"Unique idioms: {len(idiom_counts)}")
print()
print("Most frequent idioms:")
for idiom, count in idiom_counts.most_common(20):
    print(f"  {idiom:<35} {count:>4}")

Unique idioms: 1756

Most frequent idioms:
  in the long run                      208
  play games                           199
  on the level                         198
  on board                             197
  in light of                          188
  in someone's pocket                  187
  on edge                              182
  in business                          181
  down the road                        181
  bear in mind                         181
  at sea                               181
  as a rule                            180
  on the rocks                         180
  come to terms with                   179
  with a view to                       178
  in the long term                     177
  at the end of the day                177
  way to go                            177
  all over the place                   177
  in the works                         176


In [37]:
# Least frequent idioms (rare / harder to detect)
print("Rarest idioms (1-2 instances):")
for idiom, count in idiom_counts.most_common()[-20:]:
    print(f"  {idiom:<35} {count:>4}")

Rarest idioms (1-2 instances):
  haul someone over the coals            1
  bedroom eyes                           1
  there's no fool like an old fool       1
  lock the stable door after the horse has bolted    1
  piss and vinegar                       1
  nutty as a fruitcake                   1
  cruising for a bruising                1
  thumb your nose at                     1
  meet your maker                        1
  you can't teach an old dog new tricks    1
  pardon my French                       1
  have a tiger by the tail               1
  quick on the draw                      1
  get a rise out of                      1
  walk the talk                          1
  think outside the box                  1
  beat the rap                           1
  meet your Waterloo                     1
  been there, done that                  1
  over the transom                       1


In [ ]:
# Show idiomatic vs literal uses of the same expression
example_idiom = 'break the ice'
examples = [e for e in magpie if e['idiom'] == example_idiom]
print(f"'{example_idiom}' — {len(examples)} instances")
print()

for label_code in ['i', 'l']:
    subset = [e for e in examples if e['label'] == label_code]
    name = label_names.get(label_code, label_code)
    print(f"  {name.upper()} uses ({len(subset)}):")
    for e in subset[:2]:
        # context is a list of sentences, the middle one contains the idiom
        ctx = [s for s in e['context'] if s.strip()]
        if ctx:
            print(f"    \"{ctx[0][:120]}...\"")
            print(f"    (confidence: {e['confidence']:.2f})")
    print()

'screw' — 0 instances

  IDIOMATIC uses (0):

  LITERAL uses (0):



In [38]:
# Genre distribution
genres = Counter(e['genre'] for e in magpie)
print("Genre distribution:")
for genre, count in genres.most_common():
    print(f"  {genre:<25} {count:>6}")

Genre distribution:
  W fict prose               13250
  W pop lore                  5797
  W misc                      4416
  S conv                      2849
  PMB                         2358
  W biography                 2107
  W newsp other: report       1985
  W nonAc: soc science        1458
  W commerce                  1452
  W nonAc: humanities arts    1330
  W nonAc: polit law edu      1219
  W ac:soc science            1160
  W newsp other: sports       1121
  W ac:polit law edu          1081
  W news script               1077
  W newsp other: social       1071
  W ac:humanities arts        1046
  W nonAc: nat science        1022
  W newsp tabloid              883
  W newsp brdsht nat: misc     769
  S meeting                    748
  W nonAc: tech engin          642
  W religion                   627
  S interview oral history     512
  W hansard                    478
  S brdcast discussn           429
  W newsp brdsht nat: report    330
  S speech unscripted          311

In [26]:
# Confidence distribution
import statistics

confidences = [e['confidence'] for e in magpie]
print(f"Confidence stats:")
print(f"  Mean:   {statistics.mean(confidences):.3f}")
print(f"  Median: {statistics.median(confidences):.3f}")
print(f"  Stdev:  {statistics.stdev(confidences):.3f}")
print(f"  Min:    {min(confidences):.3f}")
print(f"  Max:    {max(confidences):.3f}")

conf_buckets = Counter()
for c in confidences:
    if c >= 0.9: conf_buckets['0.9-1.0'] += 1
    elif c >= 0.7: conf_buckets['0.7-0.9'] += 1
    elif c >= 0.5: conf_buckets['0.5-0.7'] += 1
    else: conf_buckets['<0.5'] += 1

print("\nConfidence buckets:")
for bucket in ['0.9-1.0', '0.7-0.9', '0.5-0.7', '<0.5']:
    count = conf_buckets.get(bucket, 0)
    bar = '#' * (count // 200)
    print(f"  {bucket:<10} {count:>6}  {bar}")

Confidence stats:
  Mean:   0.933
  Median: 1.000
  Stdev:  0.137
  Min:    0.146
  Max:    1.000

Confidence buckets:
  0.9-1.0     44488  ##############################################################################################################################################################################################################################
  0.7-0.9      8378  #########################################
  0.5-0.7      2876  ##############
  <0.5          880  ####


In [27]:
# Variant types
variants = Counter(e.get('variant_type', 'standard') for e in magpie)
print("Variant types:")
for vtype, count in variants.most_common():
    print(f"  {str(vtype):<20} {count:>6}")

Variant types:
  identical             26375
  inflection             9305
  combined-inflection    6523
  dashes                 3580
  case                   2049
  combined-other         1919
  insertion-other        1830
  misc                   1416
  determiner-broad        971
  deletion-determiner     777
  possessive              653
  determiner-narrow       592
  insertion-determiner    457
  objective               175


---
## 5. Cross-Dataset Overlap

How much overlap is there between the datasets?

In [28]:
# Normalize all phrasal verbs / idioms for comparison
semi_phrases = set(
    e['verb'].replace('*', '').replace('+', '').strip().lower()
    for e in semi
)
wecan_phrases = set(k.lower() for k in wecan.keys())
mcgraw_phrases = set(e['phrase'].lower() for e in mcgraw)
magpie_idioms = set(e['idiom'].lower() for e in magpie)

print(f"Semigradsky:  {len(semi_phrases):>6} unique")
print(f"WeCanPhrasal: {len(wecan_phrases):>6} unique")
print(f"McGraw-Hill:  {len(mcgraw_phrases):>6} unique")
print(f"MAGPIE:       {len(magpie_idioms):>6} unique")

Semigradsky:      84 unique
WeCanPhrasal:   3350 unique
McGraw-Hill:   21253 unique
MAGPIE:         1756 unique


In [29]:
# Overlap between phrasal verb datasets
pv_overlap = semi_phrases & wecan_phrases
print(f"Semigradsky ∩ WeCanPhrasal: {len(pv_overlap)} overlap")
print(f"Only in Semigradsky:        {len(semi_phrases - wecan_phrases)}")
print(f"Only in WeCanPhrasal:       {len(wecan_phrases - semi_phrases)}")

print("\nIn Semigradsky but NOT in WeCanPhrasal:")
for pv in sorted(semi_phrases - wecan_phrases)[:10]:
    print(f"  {pv}")

Semigradsky ∩ WeCanPhrasal: 55 overlap
Only in Semigradsky:        29
Only in WeCanPhrasal:       3295

In Semigradsky but NOT in WeCanPhrasal:
  blow  up
  break  down
  bring  up
  call  off
  carry  out
  carry on about
  check in/into
  get  on
  get  up
  get on /


In [30]:
# Overlap between MAGPIE idioms and McGraw-Hill
magpie_in_mcgraw = sum(1 for idiom in magpie_idioms if idiom in mcgraw_phrases)
print(f"MAGPIE idioms found in McGraw-Hill: {magpie_in_mcgraw}/{len(magpie_idioms)} ({magpie_in_mcgraw*100/len(magpie_idioms):.0f}%)")

# Show MAGPIE idioms NOT in McGraw-Hill
not_in_mcgraw = sorted(magpie_idioms - mcgraw_phrases)
print(f"\nMAGPIE idioms NOT in McGraw-Hill ({len(not_in_mcgraw)}):")
for idiom in not_in_mcgraw[:15]:
    print(f"  {idiom}")
if len(not_in_mcgraw) > 15:
    print(f"  ... and {len(not_in_mcgraw) - 15} more")

MAGPIE idioms found in McGraw-Hill: 617/1756 (35%)

MAGPIE idioms NOT in McGraw-Hill (1139):
  a hair's breadth
  a little bird told me
  a penny for your thoughts
  a pretty penny
  above board
  act of god
  against the grain
  ahead of the curve
  ahead of the game
  all along
  all bets are off
  all in a day's work
  all over bar the shouting
  all over the map
  all over the place
  ... and 1124 more


In [31]:
# WeCanPhrasal verbs that appear in McGraw-Hill
pv_in_mcgraw = wecan_phrases & mcgraw_phrases
print(f"WeCanPhrasal verbs also in McGraw-Hill: {len(pv_in_mcgraw)}/{len(wecan_phrases)}")
print("\nSamples:")
for pv in sorted(pv_in_mcgraw)[:10]:
    print(f"  {pv}")

WeCanPhrasal verbs also in McGraw-Hill: 507/3350

Samples:
  act out
  act up
  back up
  balance out
  beam up
  belt up
  bend down
  bend over
  black out
  bliss out


---
## 6. Combined Lexicon Size

What's the total coverage if we merge all datasets?

In [32]:
all_expressions = semi_phrases | wecan_phrases | mcgraw_phrases | magpie_idioms
print(f"Total unique expressions across all datasets: {len(all_expressions):,}")
print()

# Break down by type (rough heuristic: 1-2 words = phrasal verb, 3+ = idiom)
short = [e for e in all_expressions if len(e.split()) <= 2]
medium = [e for e in all_expressions if 3 <= len(e.split()) <= 5]
long_ = [e for e in all_expressions if len(e.split()) > 5]

print(f"1-2 words (likely phrasal verbs): {len(short):>6}")
print(f"3-5 words (short idioms):         {len(medium):>6}")
print(f"6+  words (long idioms/phrases):   {len(long_):>6}")

Total unique expressions across all datasets: 25,261

1-2 words (likely phrasal verbs):   4924
3-5 words (short idioms):          15660
6+  words (long idioms/phrases):     4677


In [34]:
print("Done! All lexicons explored.")

Done! All lexicons explored.


### How would the combined dataset look like?